# Lista duplamente encadeada — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

A aula passada terminou com uma limitação: para mexer num nó é preciso o **anterior**, e a lista só
anda para a frente. Este tutorial retoma o programa `dupla.c` visto hoje — a mesma lista em ordem
crescente, agora com um ponteiro `ant` em cada nó e um registro `Lista` que guarda o primeiro e o
último — e segue o caminho dos slides: o nó com duas setas, inserir, remover dado o nó, o nó
sentinela e o custo de tudo isso. Depois de cada bloco há um exercício com teste automático; no fim
vêm as tarefas de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Declarar um nó com `ant` e `prox` e verificar a regra que mantém a lista consistente
- Inserir um nó atualizando os **quatro** ponteiros envolvidos, inclusive `ini` e `fim`
- Remover um nó a partir dele mesmo, sem percorrer a lista para achar o anterior
- Usar um nó sentinela (cabeça) para escrever inserção e remoção sem nenhum `if`
- Comparar o custo de tempo e de memória da lista simples e da dupla

In [ ]:
# Confira se o gcc está disponível e defina a função que testa os exercícios
!gcc --version | head -1

import subprocess

def testa(comando, esperado, entrada=None, tempo=20):
    '''Roda o comando e compara a saída com o esperado.'''
    try:
        r = subprocess.run(comando, shell=True, capture_output=True, text=True,
                           input=entrada, timeout=tempo)
    except subprocess.TimeoutExpired:
        print(f'Demorou mais de {tempo} s: a função está percorrendo a lista à toa?')
        return
    limpa = lambda t: '\n'.join(l.rstrip() for l in t.strip().split('\n'))
    saida, esperado = limpa(r.stdout), limpa(esperado)
    if r.returncode != 0:
        print(f'O programa terminou com erro (status {r.returncode}; '
              '-11 ou 139 = segmentation fault).')
        if r.stderr:
            print(r.stderr[:1500])
    if saida == esperado:
        print('OK!')
    else:
        print('Saída esperada:')
        print(esperado)
        print('Saída obtida:')
        print(saida)

## O programa da aula

Recompile o programa condutor **exatamente como saiu da aula**. Antes de executar, escreva a saída
que você espera: as duas linhas de cima (ida e volta) e as duas de baixo, depois de remover o 20 e
o último nó.

In [ ]:
%%writefile dupla.c
/* dupla.c --- lista duplamente encadeada em ordem crescente */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *ant;    /* no anterior, ou NULL */
    struct no *prox;   /* proximo no, ou NULL */
} No;

typedef struct {
    No *ini;   /* primeiro no, ou NULL */
    No *fim;   /* ultimo no, ou NULL */
} Lista;

/* devolve o primeiro no com valor x, ou NULL */
No *busca(const Lista *l, int x)
{
    No *p = l->ini;
    while (p != NULL && p->valor != x)
        p = p->prox;
    return p;
}

/* insere x mantendo a ordem crescente */
void insere_ordenado(Lista *l, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    No *p = l->ini;               /* o novo entra antes de p */
    while (p != NULL && p->valor < x)
        p = p->prox;
    novo->prox = p;
    novo->ant = (p == NULL) ? l->fim : p->ant;
    if (novo->ant == NULL)        /* virou o primeiro */
        l->ini = novo;
    else
        novo->ant->prox = novo;
    if (p == NULL)                /* virou o ultimo */
        l->fim = novo;
    else
        p->ant = novo;
}

/* tira o no p da lista, sem procurar o anterior */
void remove_no(Lista *l, No *p)
{
    if (p->ant == NULL)
        l->ini = p->prox;
    else
        p->ant->prox = p->prox;
    if (p->prox == NULL)
        l->fim = p->ant;
    else
        p->prox->ant = p->ant;
    free(p);
}

void imprime(const Lista *l)
{
    printf("ida:  ");
    for (const No *p = l->ini; p != NULL; p = p->prox)
        printf(" %d", p->valor);
    printf("\n");
}

void imprime_volta(const Lista *l)
{
    printf("volta:");
    for (const No *p = l->fim; p != NULL; p = p->ant)
        printf(" %d", p->valor);
    printf("\n");
}

void libera(Lista *l)
{
    No *p = l->ini;
    while (p != NULL) {
        No *prox = p->prox;
        free(p);
        p = prox;
    }
    l->ini = l->fim = NULL;
}

int main(void)
{
    int entrada[] = {30, 10, 40, 20};
    Lista l = {NULL, NULL};
    for (int i = 0; i < 4; i++)
        insere_ordenado(&l, entrada[i]);
    imprime(&l);
    imprime_volta(&l);
    No *p = busca(&l, 20);
    if (p != NULL)
        remove_no(&l, p);
    remove_no(&l, l.fim);
    imprime(&l);
    imprime_volta(&l);
    libera(&l);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra dupla.c -o dupla && ./dupla

## 1. O nó com duas setas

Cada nó ganha um segundo ponteiro, `ant`, para o nó de trás. O primeiro tem `ant == NULL`, o último
tem `prox == NULL`, e o registro `Lista` guarda os dois extremos. A lista só está **consistente**
se, para todo nó `p` que tem seguinte,

$$p\texttt{->prox->ant} = p,$$

isto é, quem eu aponto para a frente aponta de volta para mim. Cada operação precisa preservar essa
regra — o compilador não confere nada disso.

O preço é memória: um ponteiro a mais por nó. Confira quanto ocupa cada tipo de nó nesta máquina.

In [ ]:
%%writefile tamanho.c
#include <stdio.h>

struct simples { int valor; struct simples *prox; };
struct dupla   { int valor; struct dupla *ant, *prox; };

int main(void)
{
    printf("simples: %zu bytes\n", sizeof(struct simples));
    printf("dupla:   %zu bytes\n", sizeof(struct dupla));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra tamanho.c -o tamanho && ./tamanho

Um `int` ocupa 4 bytes e cada ponteiro, 8; por que então o nó simples tem 16 bytes, e não 12? (Lembre
do alinhamento visto na aula de registros.)

### Exercício 1 — a lista está consistente?

Escreva `consistente`, que devolve 1 se a lista respeita **todas** as regras abaixo e 0 caso
contrário:

- lista vazia: `ini` e `fim` são ambos `NULL`;
- `ini->ant == NULL` e `fim->prox == NULL`;
- para todo nó `p` com seguinte, `p->prox->ant == p`;
- andando pelos `prox` a partir de `ini`, o último nó alcançado é `fim`.

O `main` monta à mão (sem `malloc`) listas certas e quebradas e imprime o veredito de cada uma.

In [ ]:
%%writefile ex1.c
#include <stdio.h>

typedef struct no {
    int valor;
    struct no *ant;
    struct no *prox;
} No;

typedef struct {
    No *ini;
    No *fim;
} Lista;

/* liga os n nos de v numa lista correta, na ordem do vetor */
void monta(Lista *l, No v[], int n)
{
    for (int i = 0; i < n; i++) {
        v[i].ant = (i > 0) ? &v[i - 1] : NULL;
        v[i].prox = (i < n - 1) ? &v[i + 1] : NULL;
    }
    l->ini = (n > 0) ? &v[0] : NULL;
    l->fim = (n > 0) ? &v[n - 1] : NULL;
}

/* Exercicio 1: devolva 1 se a lista e consistente, 0 caso contrario.
   TODO: implemente consistente. */
int consistente(const Lista *l)
{
    /* TODO: implemente aqui */
    (void) l;
    return 1;
}

int main(void)
{
    No v[3] = {{10, NULL, NULL}, {20, NULL, NULL}, {30, NULL, NULL}};
    Lista l;

    monta(&l, v, 3);
    printf("%d ", consistente(&l));          /* correta */

    v[2].ant = &v[0];                        /* o 30 aponta de volta para o 10 */
    printf("%d ", consistente(&l));
    monta(&l, v, 3);

    l.fim = &v[1];                           /* fim no 20, mas o 20 tem seguinte */
    printf("%d ", consistente(&l));
    monta(&l, v, 3);

    v[0].ant = &v[2];                        /* o primeiro tem anterior */
    printf("%d ", consistente(&l));
    monta(&l, v, 3);

    Lista vazia = {NULL, NULL};
    printf("%d ", consistente(&vazia));

    Lista meia = {NULL, &v[0]};              /* ini NULL, fim nao */
    printf("%d ", consistente(&meia));

    monta(&l, v, 1);                         /* um so no */
    printf("%d\n", consistente(&l));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra ex1.c -o ex1
testa('./ex1', '1 0 0 0 1 0 1')

Guarde essa função: nos próximos exercícios, chamar `consistente` depois de cada operação é o jeito
mais rápido de descobrir *qual* operação quebrou a lista.

## 2. Inserir: quatro ponteiros

Para pôr `novo` imediatamente antes de `p`, quatro ponteiros mudam — dois do próprio `novo` e dois
dos vizinhos:

```c
novo->prox = p;
novo->ant  = p->ant;      /* ou l->fim, se p == NULL */
novo->ant->prox = novo;   /* ou l->ini = novo, se novo->ant == NULL */
p->ant = novo;            /* ou l->fim = novo, se p == NULL */
```

Os dois primeiros só *leem* os vizinhos; os dois últimos os *alteram*. Por isso os dois primeiros vêm
antes: depois de `p->ant = novo`, o antigo anterior de `p` já não é alcançável por ele.

O que acontece se a última linha for esquecida? A célula abaixo gera `insere_errada.c` sem ela.
Preveja as quatro linhas antes de executar.

In [ ]:
!python3 -c "import re; s=open('dupla.c').read(); s=s.replace('    else\n        p->ant = novo;\n', ''); open('insere_errada.c','w').write(s)"
!grep -n -A2 "virou o ultimo" insere_errada.c
!gcc -Wall -Wextra insere_errada.c -o insere_errada && ./insere_errada

A **ida** perdeu o 10 e a **volta** para no 30. Siga as inserções (30, 10, 40, 20): ao inserir o 10
antes do 30, o `ant` do 30 continuou `NULL`. Quando chega o 20, que também entra antes do 30, a
função lê `p->ant` — `NULL` — e conclui que o 20 vai ser o primeiro: `l->ini = novo`. O 10 fica
inacessível (e vaza). Na lista dupla, os `ant` não são enfeite: o próprio código os **usa**, e um
`ant` errado corrompe as operações seguintes.

### Exercício 2 — inserir nos extremos sem percorrer

Com `ini` e `fim` à mão, inserir no começo ou no fim não precisa de laço nenhum. Escreva
`insere_inicio` e `insere_fim`. O teste insere 300 000 valores no fim: uma versão que percorre a
lista não termina a tempo.

In [ ]:
%%writefile ex2.c
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *ant;
    struct no *prox;
} No;

typedef struct {
    No *ini;
    No *fim;
} Lista;

/* --- funcoes da aula (nao precisa mexer) --- */
No *novo_no(int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) { perror("malloc"); exit(1); }
    novo->valor = x;
    novo->ant = novo->prox = NULL;
    return novo;
}

void imprime(const Lista *l)
{
    printf("ida:  ");
    for (const No *p = l->ini; p != NULL; p = p->prox)
        printf(" %d", p->valor);
    printf("\n");
}

void imprime_volta(const Lista *l)
{
    printf("volta:");
    for (const No *p = l->fim; p != NULL; p = p->ant)
        printf(" %d", p->valor);
    printf("\n");
}

void libera(Lista *l)
{
    No *p = l->ini;
    while (p != NULL) {
        No *prox = p->prox;
        free(p);
        p = prox;
    }
    l->ini = l->fim = NULL;
}

/* Exercicio 2: insira x no inicio / no fim, sem percorrer a lista.
   Nao esqueca da lista vazia.
   TODO: implemente insere_inicio e insere_fim. */
void insere_inicio(Lista *l, int x)
{
    No *novo = novo_no(x);
    /* TODO: implemente aqui */
    (void) l;
    free(novo);
}

void insere_fim(Lista *l, int x)
{
    No *novo = novo_no(x);
    /* TODO: implemente aqui */
    (void) l;
    free(novo);
}

int main(void)
{
    Lista l = {NULL, NULL};
    insere_fim(&l, 3);
    insere_fim(&l, 4);
    insere_inicio(&l, 2);
    insere_fim(&l, 5);
    insere_inicio(&l, 1);
    imprime(&l);
    imprime_volta(&l);
    libera(&l);

    insere_inicio(&l, 7);                    /* inicio na lista vazia */
    imprime_volta(&l);
    libera(&l);

    for (int i = 0; i < 300000; i++)
        insere_fim(&l, i);
    printf("%d %d\n", l.ini ? l.ini->valor : -1, l.fim ? l.fim->valor : -1);
    libera(&l);
    return 0;
}

In [ ]:
!gcc -O2 -Wall -Wextra ex2.c -o ex2
testa('./ex2', '''ida:   1 2 3 4 5
volta: 5 4 3 2 1
volta: 7
0 299999''', tempo=5)

## 3. Remover dado o nó

Na lista simples, para tirar um nó era preciso andar com `ant` um passo atrás, porque só o anterior
sabe onde o nó está. Agora o próprio nó sabe: `p->ant`. Remover é ligar os dois vizinhos de `p`
entre si e liberar `p`, **sem laço nenhum**:

```c
p->ant->prox = p->prox;   /* ou l->ini = p->prox, se p e o primeiro */
p->prox->ant = p->ant;    /* ou l->fim = p->ant,  se p e o ultimo   */
free(p);
```

Os dois casos especiais são os extremos — e cada um mexe num campo diferente do registro `Lista`.
A célula abaixo gera `remove_errada.c`, que esquece de atualizar `l->fim`. O `main` remove o
último nó e depois imprime a volta. O que você espera?

In [ ]:
!python3 -c "s=open('dupla.c').read(); s=s.replace('    if (p->prox == NULL)\n        l->fim = p->ant;\n    else\n        p->prox->ant = p->ant;\n', '    if (p->prox != NULL)\n        p->prox->ant = p->ant;\n'); open('remove_errada.c','w').write(s)"
!grep -n -B6 -A2 "free(p);" remove_errada.c | head -12
!gcc -Wall -Wextra remove_errada.c -o remove_errada && ./remove_errada; echo "status: $?"
!gcc -g -fsanitize=address remove_errada.c -o remove_errada-asan && ./remove_errada-asan 2>&1 | head -18

`l->fim` continua apontando para o 40, que acabou de ser liberado: `imprime_volta` começa a andar
por um bloco que já não pertence ao programa. Sem o *sanitizer*, o resultado depende do que o
`malloc` fez com aquele bloco — no terminal o programa costuma cair logo depois da terceira linha.
Com `-fsanitize=address`, o relatório diz exatamente o que houve: **heap-use-after-free**, lido em
`imprime_volta`, liberado em `remove_no`, alocado em `insere_ordenado`.

### Exercício 3 — remover todas as ocorrências

Escreva `remove_todos`, que tira da lista **todos** os nós com valor `x`, usando a `remove_no` da
aula. Atenção: depois de `remove_no(l, p)`, o nó `p` não existe mais — `p->prox` não pode ser lido.

In [ ]:
%%writefile ex3.c
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *ant;
    struct no *prox;
} No;

typedef struct {
    No *ini;
    No *fim;
} Lista;

/* --- funcoes da aula (nao precisa mexer) --- */
void insere_fim(Lista *l, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) { perror("malloc"); exit(1); }
    novo->valor = x;
    novo->prox = NULL;
    novo->ant = l->fim;
    if (l->fim == NULL)
        l->ini = novo;
    else
        l->fim->prox = novo;
    l->fim = novo;
}

void remove_no(Lista *l, No *p)
{
    if (p->ant == NULL)
        l->ini = p->prox;
    else
        p->ant->prox = p->prox;
    if (p->prox == NULL)
        l->fim = p->ant;
    else
        p->prox->ant = p->ant;
    free(p);
}

void imprime(const Lista *l)
{
    printf("ida:  ");
    for (const No *p = l->ini; p != NULL; p = p->prox)
        printf(" %d", p->valor);
    printf("\n");
}

void imprime_volta(const Lista *l)
{
    printf("volta:");
    for (const No *p = l->fim; p != NULL; p = p->ant)
        printf(" %d", p->valor);
    printf("\n");
}

void libera(Lista *l)
{
    No *p = l->ini;
    while (p != NULL) {
        No *prox = p->prox;
        free(p);
        p = prox;
    }
    l->ini = l->fim = NULL;
}

/* Exercicio 3: remova todos os nos com valor x.
   TODO: implemente remove_todos (use remove_no). */
void remove_todos(Lista *l, int x)
{
    /* TODO: implemente aqui */
    (void) l;
    (void) x;
}

int main(void)
{
    int v[] = {10, 10, 20, 10, 30, 10};
    Lista l = {NULL, NULL};
    for (int i = 0; i < 6; i++)
        insere_fim(&l, v[i]);
    remove_todos(&l, 10);
    imprime(&l);
    imprime_volta(&l);
    remove_todos(&l, 99);                    /* nao esta na lista */
    imprime(&l);
    libera(&l);

    for (int i = 0; i < 3; i++)
        insere_fim(&l, 7);
    remove_todos(&l, 7);                     /* todos */
    imprime(&l);
    imprime_volta(&l);
    libera(&l);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex3.c -o ex3
testa('./ex3', '''ida:   20 30
volta: 30 20
ida:   20 30
ida:
volta:''')

## 4. O nó sentinela

Todos os `if` das funções anteriores existem pelo mesmo motivo: o primeiro nó não tem anterior e o
último não tem seguinte — a seta que precisaria mudar mora fora dos nós, em `l->ini` ou `l->fim`.
A saída é a mesma ideia do `No **` da aula passada, levada um passo adiante: criar um nó **extra**,
a *cabeça* (ou sentinela), que não guarda valor e fecha a lista num círculo:

- `cab->prox` é o primeiro nó e `cab->ant` é o último;
- a lista vazia é a cabeça apontando para si mesma nos dois sentidos;
- percorrer é andar até **voltar à cabeça**, em vez de até `NULL`.

Como todo nó de verdade tem anterior e seguinte, `insere_antes` e `remove_no` ficam com quatro e duas
linhas, sem nenhum caso especial.

In [ ]:
%%writefile sentinela.c
/* sentinela.c --- lista dupla circular com no cabeca: sem casos especiais */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *ant;
    struct no *prox;
} No;

/* a cabeca nao guarda valor: cab->prox e o primeiro, cab->ant o ultimo */
void inicia(No *cab)
{
    cab->ant = cab->prox = cab;
}

/* insere x imediatamente antes do no p */
void insere_antes(No *p, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    novo->prox = p;
    novo->ant = p->ant;
    p->ant->prox = novo;
    p->ant = novo;
}

void insere_ordenado(No *cab, int x)
{
    No *p = cab->prox;
    while (p != cab && p->valor < x)
        p = p->prox;
    insere_antes(p, x);
}

void remove_no(No *p)
{
    p->ant->prox = p->prox;
    p->prox->ant = p->ant;
    free(p);
}

No *busca(No *cab, int x)
{
    No *p = cab->prox;
    while (p != cab && p->valor != x)
        p = p->prox;
    return (p == cab) ? NULL : p;
}

void imprime(const No *cab)
{
    printf("ida:  ");
    for (const No *p = cab->prox; p != cab; p = p->prox)
        printf(" %d", p->valor);
    printf("\n");
}

void imprime_volta(const No *cab)
{
    printf("volta:");
    for (const No *p = cab->ant; p != cab; p = p->ant)
        printf(" %d", p->valor);
    printf("\n");
}

void libera(No *cab)
{
    while (cab->prox != cab)
        remove_no(cab->prox);
}

int main(void)
{
    int entrada[] = {30, 10, 40, 20};
    No cab;
    inicia(&cab);
    for (int i = 0; i < 4; i++)
        insere_ordenado(&cab, entrada[i]);
    imprime(&cab);
    imprime_volta(&cab);
    No *p = busca(&cab, 20);
    if (p != NULL)
        remove_no(p);
    remove_no(cab.ant);
    imprime(&cab);
    imprime_volta(&cab);
    libera(&cab);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra sentinela.c -o sentinela && ./sentinela

A saída é idêntica à de `dupla.c`. Repare também em `libera`: ela só repete “remova o primeiro” até a
lista ficar vazia (`cab->prox == cab`). E a cabeça é uma variável local do `main` — ela não vem do
`malloc` e não deve ir para o `free`.

### Exercício 4 — inverter a lista

Na lista com sentinela, inverter a ordem não exige mover nenhum valor: basta **trocar `ant` e
`prox` em todos os nós — inclusive na cabeça**. Escreva `inverte`. Por que a cabeça também precisa
ser trocada?

In [ ]:
%%writefile ex4.c
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *ant;
    struct no *prox;
} No;

/* --- funcoes da aula (nao precisa mexer) --- */
void inicia(No *cab)
{
    cab->ant = cab->prox = cab;
}

void insere_antes(No *p, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) { perror("malloc"); exit(1); }
    novo->valor = x;
    novo->prox = p;
    novo->ant = p->ant;
    p->ant->prox = novo;
    p->ant = novo;
}

void remove_no(No *p)
{
    p->ant->prox = p->prox;
    p->prox->ant = p->ant;
    free(p);
}

void imprime(const No *cab)
{
    printf("ida:  ");
    for (const No *p = cab->prox; p != cab; p = p->prox)
        printf(" %d", p->valor);
    printf("\n");
}

void imprime_volta(const No *cab)
{
    printf("volta:");
    for (const No *p = cab->ant; p != cab; p = p->ant)
        printf(" %d", p->valor);
    printf("\n");
}

void libera(No *cab)
{
    while (cab->prox != cab)
        remove_no(cab->prox);
}

/* Exercicio 4: inverta a lista trocando ant e prox de todos os nos,
   inclusive da cabeca.
   TODO: implemente inverte. */
void inverte(No *cab)
{
    /* TODO: implemente aqui */
    (void) cab;
}

int main(void)
{
    No cab;
    inicia(&cab);
    for (int v = 10; v <= 40; v += 10)
        insere_antes(&cab, v);               /* antes da cabeca = no fim */
    inverte(&cab);
    imprime(&cab);
    imprime_volta(&cab);
    insere_antes(&cab, 5);                   /* continua inserindo no fim? */
    imprime(&cab);
    libera(&cab);

    inverte(&cab);                           /* lista vazia */
    imprime(&cab);
    insere_antes(&cab, 1);
    inverte(&cab);                           /* um so no */
    imprime_volta(&cab);
    libera(&cab);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex4.c -o ex4
testa('./ex4', '''ida:   40 30 20 10
volta: 10 20 30 40
ida:   40 30 20 10 5
ida:
volta: 1''')

## 5. Quanto custa

A lista dupla ganha nas operações que precisam do **anterior**: remover um nó dado, remover o último,
andar para trás. O programa abaixo monta $n$ nós e depois remove sempre o último, até esvaziar, nas
duas listas. Na simples, cada remoção procura o penúltimo; na dupla, ele é `fim->ant`.

In [ ]:
%%writefile remove_fim.c
/* remove_fim.c --- montar n nos e remover sempre o ultimo: simples x dupla */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

typedef struct simples {
    int valor;
    struct simples *prox;
} Simples;

typedef struct dupla {
    int valor;
    struct dupla *ant, *prox;
} Dupla;

static double agora(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + t.tv_nsec / 1e9;
}

static void *aloca(size_t tam)
{
    void *p = malloc(tam);
    if (p == NULL) {
        perror("malloc");
        exit(1);
    }
    return p;
}

/* simples: para tirar o ultimo, e preciso achar o penultimo */
Simples *remove_fim_simples(Simples *ini)
{
    if (ini->prox == NULL) {
        free(ini);
        return NULL;
    }
    Simples *p = ini;
    while (p->prox->prox != NULL)
        p = p->prox;
    free(p->prox);
    p->prox = NULL;
    return ini;
}

/* dupla: o penultimo e fim->ant */
void remove_fim_dupla(Dupla **ini, Dupla **fim)
{
    Dupla *p = *fim;
    *fim = p->ant;
    if (*fim == NULL)
        *ini = NULL;
    else
        (*fim)->prox = NULL;
    free(p);
}

int main(int argc, char *argv[])
{
    int n = (argc > 1) ? atoi(argv[1]) : 20000;

    Simples *s = NULL;
    for (int i = 0; i < n; i++) {
        Simples *novo = aloca(sizeof *novo);
        novo->valor = i;
        novo->prox = s;
        s = novo;
    }
    double t0 = agora();
    while (s != NULL)
        s = remove_fim_simples(s);
    double t_simples = agora() - t0;

    Dupla *ini = NULL, *fim = NULL;
    for (int i = 0; i < n; i++) {
        Dupla *novo = aloca(sizeof *novo);
        novo->valor = i;
        novo->ant = fim;
        novo->prox = NULL;
        if (fim == NULL)
            ini = novo;
        else
            fim->prox = novo;
        fim = novo;
    }
    t0 = agora();
    while (fim != NULL)
        remove_fim_dupla(&ini, &fim);
    double t_dupla = agora() - t0;

    printf("n = %d: simples %.4f s, dupla %.4f s\n", n, t_simples, t_dupla);
    printf("sizeof: simples %zu bytes, dupla %zu bytes\n",
           sizeof(Simples), sizeof(Dupla));
    return 0;
}

In [ ]:
# Visualização: remover sempre o último, lista simples x dupla
import matplotlib.pyplot as plt

!gcc -O2 -Wall -Wextra remove_fim.c -o remove_fim

ns, t_s, t_d = [], [], []
for n in [5000, 10000, 20000, 30000, 40000, 60000]:
    s = subprocess.run(['./remove_fim', str(n)], capture_output=True, text=True).stdout
    partes = s.split('\n')[0].replace(',', '').split()   # n = N: simples X s dupla Y s
    ns.append(n)
    t_s.append(float(partes[4]))
    t_d.append(float(partes[7]))
print(s.split('\n')[1])

plt.plot(ns, t_s, 'o-', label='simples (procura o penúltimo)')
plt.plot(ns, t_d, 's-', label='dupla (fim->ant)')
plt.xlabel('n (nós removidos)')
plt.ylabel('tempo (s)')
plt.title('Remover sempre o último nó')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

A curva da simples é a mesma parábola da aula passada: $(n-1) + (n-2) + \cdots + 1 \approx n^2/2$
passos. A da dupla é praticamente uma reta no zero — cada remoção custa um número fixo de
operações. O preço foi pago na memória: 24 em vez de 16 bytes por nó, 50% a mais, e quatro
ponteiros (não dois) para manter certos a cada inserção.

| operação | simples (`ini`) | simples (`ini`, `fim`) | dupla (`ini`, `fim`) |
|---|---|---|---|
| inserir no início | constante | constante | constante |
| inserir no fim | $n$ | constante | constante |
| remover o primeiro | constante | constante | constante |
| remover o último | $n$ | $n$ | constante |
| remover um nó dado | $n$ | $n$ | constante |
| buscar um valor | $n$ | $n$ | $n$ |
| percorrer de trás para a frente | — | — | $n$ |

A busca não melhora: achar um valor ainda exige olhar os nós um a um.

## Tarefas: altere o programa da aula

Volte à célula de `dupla.c`, faça cada alteração, reexecute e confira se a saída bate com o que
você previu.

1. **Esvaziar pelos dois lados.** Troque as duas remoções do `main` por um laço que remove
   alternadamente `l.ini` e `l.fim` até a lista ficar vazia, imprimindo ida e volta a cada passo.
   Quais linhas de `remove_no` são executadas na última remoção?
2. **Buscar de trás para a frente.** Escreva `busca_volta`, que começa em `l->fim` e anda pelos
   `ant`. Para que valores de `x` ela visita menos nós que `busca`? Numa lista em ordem crescente,
   como escolher entre as duas sem percorrer nada?
3. **Valores repetidos.** Insira o 20 uma segunda vez. O novo 20 entra antes ou depois do antigo?
   Troque `<` por `<=` no laço de `insere_ordenado` e preveja o que muda — imprima o endereço de
   cada nó com `%p` para conferir.

## Desafio — um editor de linha com cursor

Um editor de texto guarda a linha sendo editada e a posição do **cursor**. Escrever, apagar e mover
o cursor para os dois lados são exatamente as operações em que a lista dupla é boa. Complete o
programa abaixo: os caracteres ficam numa lista dupla circular com sentinela, e o cursor é um
ponteiro para o nó **à direita** dele (a própria cabeça, se o cursor está no fim da linha).

| comando | efeito |
|---|---|
| `i texto` | insere os caracteres de `texto` à esquerda do cursor (o cursor fica depois deles) |
| `<` | move o cursor uma posição para a esquerda (se possível) |
| `>` | move o cursor uma posição para a direita (se possível) |
| `x` | apaga o caractere à esquerda do cursor, como o *backspace* (se houver) |
| `=` | imprime a linha com um `\|` na posição do cursor |

Uma linha com vários `<` ou `>` seguidos, como `<<<`, move o cursor uma vez para cada um. Nenhuma
operação pode percorrer a linha. Com `-fsanitize=address`, nenhum erro.

In [ ]:
%%writefile editor.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

typedef struct no {
    char c;
    struct no *ant;
    struct no *prox;
} No;

typedef struct {
    No cab;      /* sentinela: cab.prox e o 1o caractere, cab.ant o ultimo */
    No *cursor;  /* no a direita do cursor (&cab se o cursor esta no fim) */
} Editor;

void inicia(Editor *e)
{
    e->cab.ant = e->cab.prox = &e->cab;
    e->cursor = &e->cab;
}

/* TODO: insere o caractere c a esquerda do cursor */
void escreve(Editor *e, char c)
{
    (void) e;
    (void) c;
}

/* TODO: move o cursor para a esquerda, se ele nao estiver no inicio */
void esquerda(Editor *e)
{
    (void) e;
}

/* TODO: move o cursor para a direita, se ele nao estiver no fim */
void direita(Editor *e)
{
    (void) e;
}

/* TODO: apaga o caractere a esquerda do cursor, se houver */
void apaga(Editor *e)
{
    (void) e;
}

/* TODO: imprime a linha com '|' na posicao do cursor */
void mostra(const Editor *e)
{
    (void) e;
    printf("\n");
}

/* TODO: libera todos os nos (a cabeca nao veio do malloc) */
void libera(Editor *e)
{
    (void) e;
}

int main(void)
{
    Editor e;
    inicia(&e);
    char linha[256];
    while (fgets(linha, sizeof linha, stdin) != NULL) {
        linha[strcspn(linha, "\n")] = '\0';
        if (linha[0] == 'i' && linha[1] == ' ') {
            for (const char *s = linha + 2; *s != '\0'; s++)
                escreve(&e, *s);
        } else if (linha[0] == 'x') {
            apaga(&e);
        } else if (linha[0] == '=') {
            mostra(&e);
        } else {
            for (const char *s = linha; *s != '\0'; s++) {
                if (*s == '<')
                    esquerda(&e);
                else if (*s == '>')
                    direita(&e);
            }
        }
    }
    libera(&e);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra editor.c -o editor
entrada = '''i progama
=
<<<
i r
=
>>>
x
=
<<<<<<<<<<
x
i C-
=
>>>>
x
>>>>>>
=
'''
testa('./editor', '''progama|
progr|ama
program|
C-|program
C-proram|''', entrada=entrada)

Repare na entrada `<<<<<<<<<<`: são dez movimentos numa linha de sete caracteres, e o cursor
precisa parar no começo. Depois tente acrescentar dois comandos: `^` (cursor para o início) e `$`
(cursor para o fim) — quantas operações cada um custa? E um comando `d`, que apaga o caractere à
**direita** do cursor: o que acontece com `e->cursor` depois do `free`?

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Os pontos principais desta aula estão
em:

- Celes, Cerqueira e Rangel (2004), *Introdução a estruturas de dados*, cap. 10 — listas
  circulares e listas duplamente encadeadas;
- Cormen et al. (2009), *Introduction to Algorithms*, §10.2 — listas duplamente encadeadas
  (`LIST-INSERT`, `LIST-DELETE`) e sentinelas (`L.nil`);
- Knuth (1997), *The Art of Computer Programming*, vol. 1, §2.2.5 — listas duplamente encadeadas;
- Feofiloff (2009), *Algoritmos em linguagem C*, cap. 4 — listas encadeadas;
- Tenenbaum, Langsam e Augenstein (1995), *Estruturas de dados usando C*, cap. 4 — listas
  duplamente ligadas;
- Serebryany et al. (2012) — o AddressSanitizer e a detecção de *use-after-free*.